In [19]:
import pickle
import pandas as pd
from sklearn.feature_extraction import DictVectorizer
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import root_mean_squared_error
from sklearn.pipeline import make_pipeline

In [20]:
import mlflow
mlflow.set_experiment('green-taxi-duration')

<Experiment: artifact_location='file:///home/dongenxi/mlops-zoomcamp/04-deployment/web-service-mlflow2/mlruns/974322736519426442', creation_time=1788954598030, experiment_id='974322736519426442', last_update_time=1788954598030, lifecycle_stage='active', name='green-taxi-duration', tags={}>

In [21]:
def read_dataframe(filename: str):
    df = pd.read_parquet(filename)
    df['duration'] = df.lpep_dropoff_datetime - df.lpep_pickup_datetime
    df['duration'] = df['duration'].dt.total_seconds() / 60
    df = df[(df['duration'] >= 1) & (df['duration'] <= 60)]

    cat = ['PULocationID', 'DOLocationID']
    df[cat] = df[cat].astype(str)
    return df

def prepare_dicts(df: pd.DataFrame):
    df['PU_DO'] = df['PULocationID'] + '_' + df['DOLocationID']
    cat = ['PU_DO']
    num = ['trip_distance']
    dicts = df[cat + num].to_dict(orient='records')

    return dicts

In [22]:
df_train = read_dataframe('/home/dongenxi/mlops-zoomcamp/data/green_tripdata_2021-01.parquet')
df_val = read_dataframe('/home/dongenxi/mlops-zoomcamp/data/green_tripdata_2021-02.parquet')

y_train = df_train['duration'].values
y_val = df_val['duration'].values

dicts_train = prepare_dicts(df_train)
dicts_val = prepare_dicts(df_val)

In [25]:
with mlflow.start_run():

    params = dict(max_depth=20, n_estimators=100, min_samples_leaf=10, random_state=0)
    mlflow.log_params(params)

    pipeline = make_pipeline(
        DictVectorizer(),
        RandomForestRegressor(**params, n_jobs=-1)
    )

    pipeline.fit(dicts_train, y_train)
    model_info = mlflow.sklearn.log_model(sk_model=pipeline, artifact_path='model')
    y_pred = pipeline.predict(dicts_val)

    rmse = root_mean_squared_error(y_val, y_pred)
    mlflow.log_metric('rmse', rmse)
    print(rmse)

model_info

2026/09/09 12:07:06 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/09/09 12:07:10 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


6.7558229919200725


In [27]:
uri = 'runs:/622d5f916c7e4ed78a4b332fcb10a675/model'

loaded_model = mlflow.pyfunc.load_model(uri)